# In-context vs in-weights learning in a small transformer

A transformer can solve a classification problem in two ways:

- **In-weights learning (IWL):** memorise which label goes with which class, in the weights.
- **In-context learning (ICL):** look up the answer in the examples given in the prompt.

[Chan et al. (2022)](https://arxiv.org/abs/2205.05055) showed that which strategy a transformer picks depends on the *statistics of the training data*. This repo is a small version of that experiment (the task follows [Reddy, 2023](https://arxiv.org/abs/2312.03002)) that trains in about a minute on a GPU.

**The task.** Items are handwritten characters from [Omniglot](https://github.com/brendenlake/omniglot) (1623 characters from 50 alphabets, 20 drawings each; each 90° rotation counts as its own class, giving 6492 classes). A small CNN, trained once in the setup step, turns every drawing into a 64-dimensional embedding, and that embedding is what the transformer sees. Every class has a fixed label (32 labels, so classes share them). The model sees 8 (item, label) pairs followed by a query item, and has to output the query's label. An abstract version with Gaussian clusters instead of characters is available with `dataset="gaussian"`.

**Two tests, run throughout training:**

- **ICL accuracy:** brand-new classes with randomly assigned labels. The query's class appears in the context. Only a model that reads the context can get this right.
- **IWL accuracy:** trained classes, but the query's class and label are *absent* from the context. Only a model that has memorised the class can get this right.

**Two data knobs:**

- `p_bursty`: the fraction of training sequences that are "bursty", meaning the query's class appears 3 times in the context.
- `n_classes`: how many classes there are to memorise.

This notebook is a thin wrapper around the `icl_iwl` package in this repo. Set the runtime to GPU (Runtime → Change runtime type → T4) before you start.

## 0. Setup

On Colab this clones the repo and installs it (≈30 s). Locally (with `pip install -e .` done) it just moves to the repo root.

In [ ]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('icl-iwl-exercise'):
    !git clone -q https://github.com/jessegeerts/icl-iwl-exercise.git icl-iwl-exercise
    %cd icl-iwl-exercise
    !pip install -q -e .
elif os.path.basename(os.getcwd()) == 'notebooks':
    %cd ..
if not os.path.exists('data/omniglot.pt'):
    !python scripts/prepare_omniglot.py

In [ ]:
%load_ext autoreload
%autoreload 2
from icl_iwl import DEVICE, Config, Task, train, compare, plot_runs, plot_attention, plot_grid, plot_examples
print('running on', DEVICE)

## 1. The data

A *class* is an Omniglot character (at one of four rotations); an *item* is the 64-d embedding of one drawing of it, produced by the small CNN trained in the setup step (`icl_iwl/omniglot.py`). Every class has a fixed label, `class % 32`. Below: a few drawings of a few training classes.

In [ ]:
task = Task(Config(n_classes=1024))
plot_examples(task, n_classes=5, n_drawings=8)

The data, model and training loop live in `icl_iwl/`. Everything you might want to change is in `icl_iwl/config.py` (`Config`) or `icl_iwl/model.py`. Any `Config` field can be passed to `train(...)` as a keyword argument.

## 2. Baseline

A 2-layer transformer on fully bursty data with 4096 classes.

In [ ]:
model, task, log = train(n_classes=4096, p_bursty=1.0, steps=4000)
plot_runs({"baseline": [log]})

What is the model attending to? Below: how much the **query** (last position) attends to the items and labels of each class in the last layer, then the full attention pattern of every layer, averaged over a batch of ICL test sequences. Positions are ordered item, label, item, label, …, query (so even positions are items, odd positions are labels, and 16 is the query).

In [ ]:
plot_attention(model, task)

## 3. The data decides the strategy

Same model, different training data. Before running: which corner of this grid do you expect to give ICL, and which IWL?

In [ ]:
Ks, ps = [1024, 4096], [0.0, 0.5, 1.0]
grid = {(K, p): train(verbose=False, n_classes=K, p_bursty=p, steps=4000)[2] for K in Ks for p in ps}
plot_grid(grid, Ks, ps)

## 4. Your experiment: does the architecture matter?

Pick **one** change to the model. Then, **before you run anything**, write
down (in the cell at the bottom):

1. What you changed.
2. What you predict will happen to ICL and to IWL, and in which data regime.
3. *Why*: what does that component do that ICL or IWL needs?

Then run it against the baseline with `compare`, over at least 3 seeds, in at
least two data regimes (one where the baseline does ICL, one where it does
IWL). A single run tells you very
little: the switch between strategies is abrupt and its timing varies from
seed to seed.

Some options that only need a config change:

| Change | Config |
|---|---|
| One attention layer instead of two | `n_layers=1` |
| Remove the MLPs (attention-only) | `use_mlp=False` |
| No positional information | `pos_enc="none"` |
| Replace attention with recurrence | `model="lstm"` |
| More heads, wider or narrower model | `n_heads=4`, `d_model=16` |
| Regularisation | `weight_decay=0.1` |

Better still, change the model code yourself, either by editing
`icl_iwl/model.py` or by subclassing `Model` in a notebook cell and passing
`model_cls=MyModel` to `train` / `compare`: a different token format (item and label in one token), a different attention
rule, no LayerNorm, frozen embeddings, and so on. The *data* side is open too:
`n_bursty` sets how often the query's class repeats in a bursty context,
`noise` adds extra within-class variability, `zipf` makes some classes rare,
and `icl_iwl/omniglot.py` lets you change the embedder (its width, bottleneck
size, or how many drawings it trains on) and rebuild `data/omniglot.pt`. If you do, keep the baseline
reachable (e.g. behind a new `Config` flag) so that `compare` can still run
both. On Colab, open the **Files** pane on the left, go to `icl-iwl-exercise/icl_iwl/` and double-click `model.py` to edit it; the `autoreload` cell above picks up the saved file on the next `train(...)`. The template cell below shows the subclass route.


In [ ]:
# Option A: a config change.
runs = compare(
    {"baseline": {}, "my variant": {"n_layers": 1}},
    n_classes=4096, p_bursty=1.0, steps=4000,
)

In [ ]:
# Option B: a code change, without editing model.py. Subclass Model and override
# __init__ or forward. Example: remove the final LayerNorm.
import torch.nn as nn
from icl_iwl import Model

class MyModel(Model):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.ln_out = nn.Identity()

# runs = compare(
#     {"baseline": {}, "my variant": {"model_cls": MyModel}},
#     n_classes=4096, p_bursty=1.0, steps=4000,
# )

**My prediction:**

**What happened:**

**What I think this component is for:**